## Define Variables / Import MetaData

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from cartopy import crs as ccrs 
import cartopy.feature as cfeature
import pandas as pd
import hvplot.pandas
import xarray as xr
import hvplot.xarray
import geoviews.feature as gf

In [ ]:
# Define misc variables 
# I can't get the config.py to work in jupyternotebook because it does not know where $NOBACKUP is
amer_filepath = '../../ameriflux-data/'
mic_filepath = '../intermediates/'

In [ ]:
# Import site metadata csv
meta_file = amer_filepath + 'AmeriFlux-site-search-results-202410071335.tsv'
ameriflux_meta = pd.read_csv(meta_file, sep='\t')
fluxnet_meta = ameriflux_meta.loc[ameriflux_meta['AmeriFlux FLUXNET Data'] == 'Yes'] 

In [ ]:
# set map proj
proj=ccrs.PlateCarree()

## RSME plotting

In [ ]:
# Create df with lat/lons
site_subset = ['Site ID', 
               'Longitude (degrees)',
                'Latitude (degrees)',
               ]
df_meta = fluxnet_meta[site_subset]
df_meta.set_index('Site ID')

In [ ]:
# Import and merge results
results = pd.read_csv('../analysis/results.csv',index_col='SiteID')
results

In [ ]:
df = df_meta.join(results, on='Site ID')
df

In [ ]:
ds = xr.Dataset(
    coords={
        'site_id': df['Site ID'].values,
        'lat': ('site_id', df['Latitude (degrees)'].values),
        'lon': ('site_id', df['Longitude (degrees)'].values),
    }, 
    data_vars={
        'NEE_RSME': ("site_id", df['NEE_RSME'].values),
        'NPP_RSME': ("site_id", df['NPP_RSME'].values),
    }
)

In [ ]:
ds

### Xarray matplotlib

In [ ]:
cmap = plt.cm.autumn_r
cmap

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(12, 10), subplot_kw={'projection': proj}, constrained_layout=True);
fig.suptitle('MiCASA, FluxNet Sites Root Mean Squared Error (RSME)', y=0.76)
values =['NEE_RSME', 'NPP_RSME']
for ax,val in zip(axs, values):
    ax.add_feature(cfeature.COASTLINE,zorder=0)
    plot = ds.plot.scatter(x="lon", y="lat",ax=ax,
                           
                           markersize=val, edgecolor='none',add_legend=False,
                           
                            norm=colors.LogNorm(), 
                            # norm=colors.LogNorm(vmin=ds[val].min(), vmax=ds[val].max()),
                           hue=val,
                           cmap='autumn_r',
                           # cmap='cool',
                           add_colorbar=False
                          )
    
    cbar = fig.colorbar(plot, ax=ax, shrink=0.9, label=val[4:], orientation='horizontal')
    ax.set_title(val[:3])
plt.show()

### Make custom colormap with transparency

In [ ]:
# from matplotlib.colors import ListedColormap

In [ ]:
# Make transparency colormap:
cmap = plt.cm.autumn_r
cmap

In [ ]:
# cmap(np.arange(cmap.N)).shape

In [ ]:
cmap(1)

In [ ]:
# my_cmap = cmap(np.arange(cmap.N))
# my_cmap[:, -1] = np.linspace(0, 1, cmap.N)
# my_cmap = ListedColormap(my_cmap)
# my_cmap

In [ ]:
my_cmap = ListedColormap(my_cmap)
my_cmap

In [ ]:
# fig, axs = plt.subplots(1,2,figsize=(12, 10), subplot_kw={'projection': proj}, constrained_layout=True);
# fig.suptitle('MiCASA, FluxNet Sites Root Mean Squared Error (RSME)', y=0.76)
# values =['NEE_RSME', 'NPP_RSME']
# for ax,val in zip(axs, values):
#     ax.add_feature(cfeature.COASTLINE,zorder=0)
#     plot = ds.plot.scatter(x="lon", y="lat",ax=ax,
                           
#                            markersize=val, edgecolor='none',add_legend=False,
                           
#                             norm=colors.LogNorm(), 
#                             # norm=colors.LogNorm(vmin=ds[val].min(), vmax=ds[val].max()),
#                            hue=val,
#                            cmap=my_cmap,
#                            add_colorbar=False
#                           )
    
#     cbar = fig.colorbar(plot, ax=ax, shrink=0.9, label=val[4:], orientation='horizontal')
#     ax.set_title(val[:3])
# plt.show()

In [ ]:
%matplotlib ipympl
ds.plot.scatter(x="lon", y="lat")

#### Xarray bokeh plot? This doesn't work so I have to plot the dataframe

In [ ]:
# ds_NEE = ds[values[0]]
# ds_NEE

In [ ]:
# hv.extension('bokeh', inline=True)
# ds_NEE.hvplot.points(x='lon', y='lat',
#                       geo=True,
#                      # crs=proj, 
#                     # project=True
#                      )

# ds_dropped = ds_NEE.drop_indexes("site_id")
# ds_dropped = ds_dropped.drop_vars("site_id")
# ds_dropped

### Matplotlib CONUS only

In [ ]:
lons_conus = [-140, -60]
lats_conus = [20, 60]

In [ ]:
lons_conus +lats_conus

In [ ]:
fig, axs = plt.subplots(1,2,figsize=(16,10), subplot_kw={'projection': proj}, constrained_layout=True);
fig.suptitle('MiCASA, FluxNet Sites Root Mean Squared Error (RSME)', 
             y=0.7,size=16,
            )
values =['NEE_RSME', 'NPP_RSME']
for ax,val in zip(axs, values):
    ax.set_extent(lons_conus +lats_conus)
    ax.add_feature(cfeature.LAND,zorder=0, linewidth=0.3, color='lightgrey')
    ax.add_feature(cfeature.STATES,zorder=0, linewidth=0.3)
    
    plot = ds.plot.scatter(x="lon", y="lat",ax=ax,
                           
                           markersize=val, edgecolor='none',add_legend=False,
                            norm=colors.LogNorm(), 
                           hue=val,
                           cmap='autumn_r',
                           add_colorbar=False
                          )
    
    cbar = fig.colorbar(plot, ax=ax, shrink=0.8, label=val[4:], orientation='horizontal')
    ax.set_title(val[:3])
plt.show()

### Pandas Holoviews

In [ ]:
df.head()

In [ ]:
import xyzservices.providers as xyz
from matplotlib.ticker import LogFormatter

In [ ]:
min_lon, max_lon = df["Longitude (degrees)"].min(), df["Longitude (degrees)"].max()
min_lat, max_lat = df["Latitude (degrees)"].min(), df["Latitude (degrees)"].max()

print(min_lon, max_lon)
print(min_lat, max_lat)

In [ ]:
my_cmap

In [ ]:
plot_list = []
for i, value in enumerate(values): 
    plot = df.hvplot.points(x="Longitude (degrees)", 
                            y="Latitude (degrees)",
                            geo=True, 
                            crs=ccrs.PlateCarree(),
                            # projection=ccrs.PlateCarree(), # Doesn't work with tiles
    
                             #Custom cmap with transparency won't show up in bokeh
                            c=value,
                            logz=True,
                            cmap=my_cmap,
                            clabel=f'{value}',
    
                             size=45,
                             # Size values don't scale logarithmically
                            # s=values[0],
                            # scale=4500,
                             # color='red',
                            
                            tiles=True,
                            tiles_opts={'alpha': 0.4},
                            # tiles=xyz.Esri.WorldGrayCanvas,
    
    
                            hover_cols=['Site ID'],
    
                            # width=700, height=500,
                            xlim=(-170, -20),   # longitude range
                            ylim=(-60, 75),     # latitude range
                            # frame_width=800,
                            frame_height=700
                                               )
    plot_list.append(plot)

In [ ]:
(plot_list[0] * gf.coastline).opts(title="Micasa/Ameriflux Net Ecosystem Exchange (NEE) RSME")

In [ ]:
(plot_list[1] * gf.coastline).opts(title="Micasa/Ameriflux Net Primary Productivity (NPP) RSME")


## Debug

#### Try ipympl? It doesn't seem to work

In [ ]:
import ipympl

In [ ]:
%matplotlib ipympl
ds.plot.scatter(x="lon", y="lat")

In [ ]:
# !jupyter labextension list

In [ ]:
%matplotlib inline

In [ ]:
# Try to scale size by log:
# df_scale = df.copy()
# df_scale["log_NEE_RSME"] = np.log(df_scale["NEE_RSME"])
# df_scale.head()

# **** This doesn't work because the logs are negative- would have to create a pseudo log scale but this is complex ******

### Run holoviews with matplotlib backend

In [ ]:
import holoviews as hv
hv.extension('matplotlib')

In [ ]:
df.hvplot.points(x="Longitude (degrees)", 
                        y="Latitude (degrees)",
                        crs=ccrs.PlateCarree(),
                        # frame_width=800,
                        # frame_height=500
                       )

In [ ]:
plot * gf.coastline

### Run a random example

In [ ]:
import numpy as np
import xarray as xr

lat = np.linspace(-90, 90, 5)
lon = np.linspace(-180, 180, 5)
data = np.random.rand(len(lat), len(lon))

ds_ex = xr.DataArray(data, coords=[('lat', lat), ('lon', lon)], name='val')

ds_ex.hvplot.image(x='lon', y='lat')  # no geo=True yet

In [ ]:
ds_ex

In [ ]:
ds_ex.hvplot.points(x="lon", y="lat", 
                 # c=val,
                    geo=True,
                  projection=proj,
                 # coastline=True,
                 )

### Try geopandas?

In [ ]:
# import geopandas as gpd
# from shapely.geometry import Point


In [ ]:
# gdf = gpd.GeoDataFrame(
#     {"site_id": ds["site_id"].values},
#     geometry=[Point(xy) for xy in zip(ds["lon"].values, ds["lat"].values)],
#     crs="EPSG:4326"
# )